# Testing MAS for whole data management

This journal proposes a variant of implementation and testing of a general multi-agent system that executes user requests using separate workflows, which are presented as separate tools. 

In this implementation, the workflow for generating metadata became a tool for creating datasets, the workflow for adding data became a tool for adding data, and the workflow for receiving data became the corresponding tool.

## 0. Initial imports and variables

In [ ]:
import sys
sys.path.append('./../src')

In [ ]:
from dotenv import load_dotenv
import json
import os
import shutil
import yaml

from datalake_driver.drivers.local_storage_driver import LocalDriver
from data_model.datalake import Datalake

load_dotenv('./../src/.env')

In [ ]:
with open('raw_testing_material/imgs_metadata_rich_descr.json', 'r') as i_stream:
    some_dict = json.load(i_stream)

if os.path.exists('./testing_datalake'):
    shutil.rmtree('./testing_datalake')


In [ ]:
local_config = {
    'root_folder': './'
}

driver = LocalDriver(**local_config)

name = 'testing_datalake'
local_datalake = Datalake(name, name + '/', driver)

In [ ]:
local_datalake.create_dataset(
    'fluorescent_microscopy', 
    '3d fluorescent images of neurons, captured by confocal microscope', 
    some_dict
    )

local_datalake.create_dataset(
    'dendritic_spikes_9009', 
    'Dataset of 3D meshes of some dendritic spines.', 
    some_dict
    )

local_datalake.create_dataset(
    'mice_behavioral', 
    'Videos of mice with different conditions for analysing behavioral of healthy and ill mice.', 
    some_dict
    )

local_datalake.create_dataset(
    'neurons_activity', 
    'Dataset of generated neurons activities time series, captured from video.', 
    some_dict
    )

In [ ]:
MODEL_NAME = 'mistral-large-latest'
PROVIDER = 'mistralai'

CREATING_DS_PROMPT_PART = 'creating_md/chrono_creating_dataset_sp.yaml'
ADDING_PROMPT_PART = 'adding_data_to_dataset/adding_data_sp.yaml'
GETTING_PROMPT_PART = 'getting_data/getting_data_sp.yaml'
ORCHESTRATOR_PROMPT_PART = 'data_manager_orchestrator_sp.yaml'

with open(os.path.join('../src/prompts_templates/', CREATING_DS_PROMPT_PART)) as stream:
    CREATING_SP = yaml.safe_load(stream)['system_prompt']
with open(os.path.join('../src/prompts_templates/', ADDING_PROMPT_PART)) as stream:
    ADDING_SP = yaml.safe_load(stream)['system_prompt']
with open(os.path.join('../src/prompts_templates/', GETTING_PROMPT_PART)) as stream:
    GETTING_SP = yaml.safe_load(stream)['system_prompt']
with open(os.path.join('../src/prompts_templates/', ORCHESTRATOR_PROMPT_PART)) as stream:
    ORCHESTRATOR_SP = yaml.safe_load(stream)['system_prompt']

In [ ]:
ORCHESTRATOR_SP

## 1. Building elements of MAS

### 1.0 Initing datalake from readed instance 

To demonstrate the work, as in getting data demonstration, it is recommended to first run the results of the data addition demonstration - during the work of the journal, a test data lake will be created, which will be filled with data with metadata values.

### 1.1 Initing orchestrator

It is important to mention, that below code contains different tools for working wirh data lake. You can use different tools also, but it is important to descrive them into system prompt. 

In [ ]:
from mas_exec.scidatamas.orchestrator import OrchestratorAgent, OrchestratorState

dataset_manager_mas = OrchestratorAgent(system_prompt=ORCHESTRATOR_SP, model=MODEL_NAME, provider=PROVIDER)
dm_mas_wf = dataset_manager_mas.get_workflow()
dm_mas_wf = dm_mas_wf.compile()

## 2. Demonstration supervised MAS working

### 2.1 Creating dataset

In [ ]:
USER_CREARE_DS_TASK = """Hi!
I want to create a metadata schema for data that evaluates the behavior of mice in a circular arena. The dataset contains pairs of two videos: one video is obtained from a camera that shoots a mouse in a circular arena, and the second video captures the activity of neurons in the mouse's hippocampus, assessing which neurons are activated and which are not during different actions.
Such a dataset is useful for analyzing the eating of healthy and neurodegenerative disease-affected mice, comparing brain activity and their behavior. Please help me."""

working_state = OrchestratorState()
working_state['users_task'] = USER_CREARE_DS_TASK
working_state['messages'] = [('user', USER_CREARE_DS_TASK)]
res = dm_mas_wf.invoke(working_state)

In [ ]:
res['choosen_wf']

### 2.2 Getting data

In [ ]:
USER_GET_DATA_TASK = "Hi! I need some data captured by fluorescent microscope, where pixel sizes along OX and OY axies are more then 1024 an less then 3096. Can you get it from me?"

working_state = OrchestratorState()
working_state['users_task'] = USER_GET_DATA_TASK
working_state['messages'] = [('user', USER_GET_DATA_TASK)]
res = dm_mas_wf.invoke(working_state)

res['choosen_wf']

### 2.3 Adding data

In [ ]:
USER_ADD_DATA_TASK = "Hi! Add neuron images from the directory './raw_testing_material/imgs/' to my dataset fluorescent images dataset. To describe images use metadata specified in the file './raw_testing_material/imgs_meta.xml'"

working_state = OrchestratorState()
working_state['users_task'] = USER_ADD_DATA_TASK
working_state['messages'] = [('user', USER_ADD_DATA_TASK)]
res = dm_mas_wf.invoke(working_state)

res['choosen_wf']